# Notebook 05 — Cross-Encoder Reranking

**Goal:** rerank the top hybrid candidates from Notebook 04 using a Cross-Encoder.

### Pipeline
```text
User Query
    │
    ▼
Dense Retrieval + BM25
    │
    ▼
RRF
    │
    ▼
Top 10 Hybrid Candidates
    │
    ▼
Cross-Encoder
    │
    ▼
Final Top 5 Reranked Chunks
```

The Cross-Encoder receives the **query and each candidate chunk together** and produces a relevance score.

This notebook preserves the original chunk metadata so later stages can perform graph-aware context assembly, citations, and traceability.


In [1]:
# 01 — Imports and configuration

import json
import pickle
import re
from pathlib import Path

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

DATA_DIR = Path("../data")
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

INDEX_PATH = VECTOR_STORE_DIR / "quant_research.faiss"
METADATA_PATH = VECTOR_STORE_DIR / "quant_research_metadata.json"

BM25_PATH = VECTOR_STORE_DIR / "quant_research_bm25.pkl"
BM25_IDS_PATH = VECTOR_STORE_DIR / "quant_research_bm25_ids.json"

RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

DENSE_TOP_K = 20
BM25_TOP_K = 20
HYBRID_TOP_K = 10
FINAL_TOP_K = 5
RRF_K = 60

print("Reranker:", RERANKER_MODEL_NAME)
print("Hybrid candidates:", HYBRID_TOP_K)
print("Final reranked chunks:", FINAL_TOP_K)


c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reranker: BAAI/bge-reranker-base
Hybrid candidates: 10
Final reranked chunks: 5


In [2]:
# 02 — Load retrieval artifacts

assert INDEX_PATH.exists(), f"Missing FAISS index: {INDEX_PATH}"
assert METADATA_PATH.exists(), f"Missing metadata: {METADATA_PATH}"
assert BM25_PATH.exists(), f"Missing BM25 index: {BM25_PATH}"
assert BM25_IDS_PATH.exists(), f"Missing BM25 mapping: {BM25_IDS_PATH}"

dense_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

with open(BM25_IDS_PATH, "r", encoding="utf-8") as f:
    bm25_chunk_ids = json.load(f)

metadata_by_id = {m["chunk_id"]: m for m in metadata}

print("FAISS records:", dense_index.ntotal)
print("Metadata records:", len(metadata))
print("BM25 records:", len(bm25_chunk_ids))

assert dense_index.ntotal == len(metadata)
assert len(bm25_chunk_ids) == len(set(bm25_chunk_ids))


FAISS records: 1141
Metadata records: 1141
BM25 records: 1132


In [3]:
# 03 — Reconstruct BM25 corpus mapping

def tokenize_for_bm25(text):
    text = str(text).lower()
    return re.findall(r"(?u)\b\w+\b", text)

bm25_records = []
bm25_tokenized_corpus = []

for chunk_id in bm25_chunk_ids:
    record = metadata_by_id[chunk_id]
    text = record.get("retrieval_text") or record.get("text") or ""
    tokens = tokenize_for_bm25(text)

    if tokens:
        bm25_records.append(record)
        bm25_tokenized_corpus.append(tokens)

# Use the persisted BM25 object for scoring. The records above provide
# the deterministic mapping from BM25 position → chunk metadata.
assert len(bm25_records) == len(bm25_chunk_ids)

print("BM25 mapping verified:", len(bm25_records))


BM25 mapping verified: 1132


In [4]:
# 04 — Load embedding model and define dense retrieval

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def dense_retrieve(query, top_k=DENSE_TOP_K):
    query_vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = dense_index.search(
        query_vector,
        min(top_k, dense_index.ntotal),
    )

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx < 0:
            continue

        record = metadata[int(idx)]

        results.append({
            "chunk_id": record["chunk_id"],
            "score": float(score),
            "rank": rank,
        })

    return results

print("Embedding model loaded.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3685.35it/s]


Embedding model loaded.


In [5]:
# 05 — Define BM25 retrieval

def bm25_retrieve(query, top_k=BM25_TOP_K):
    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )[:top_k]

    results = []

    for rank, idx in enumerate(ranked_indices, start=1):
        results.append({
            "chunk_id": bm25_chunk_ids[idx],
            "score": float(scores[idx]),
            "rank": rank,
        })

    return results


In [6]:
# 06 — RRF hybrid retrieval

def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    top_k=HYBRID_TOP_K,
    rrf_k=RRF_K,
):
    fused = {}

    for result in dense_results:
        chunk_id = result["chunk_id"]

        fused.setdefault(
            chunk_id,
            {
                "chunk_id": chunk_id,
                "rrf_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
            },
        )

        fused[chunk_id]["dense_rank"] = result["rank"]
        fused[chunk_id]["rrf_score"] += (
            1.0 / (rrf_k + result["rank"])
        )

    for result in bm25_results:
        chunk_id = result["chunk_id"]

        fused.setdefault(
            chunk_id,
            {
                "chunk_id": chunk_id,
                "rrf_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
            },
        )

        fused[chunk_id]["bm25_rank"] = result["rank"]
        fused[chunk_id]["rrf_score"] += (
            1.0 / (rrf_k + result["rank"])
        )

    ranked = sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True,
    )

    return ranked[:top_k]


In [7]:
# 07 — Cross-Encoder model

reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Cross-Encoder loaded:", RERANKER_MODEL_NAME)


c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MD RAJA\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3722.36it/s]


Cross-Encoder loaded: BAAI/bge-reranker-base


In [8]:
# 08 — Reranking function

def rerank_candidates(query, hybrid_candidates, top_k=FINAL_TOP_K):
    pairs = []

    for candidate in hybrid_candidates:
        record = metadata_by_id[candidate["chunk_id"]]
        text = record.get("retrieval_text") or record.get("text") or ""

        pairs.append([query, text])

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(hybrid_candidates, scores):
        record = metadata_by_id[candidate["chunk_id"]]

        reranked.append({
            "chunk_id": candidate["chunk_id"],
            "rerank_score": float(score),
            "rrf_score": candidate["rrf_score"],
            "dense_rank": candidate["dense_rank"],
            "bm25_rank": candidate["bm25_rank"],
            "source_file": record.get("source_file"),
            "section_path": record.get("section_path"),
            "pages": record.get("pages"),
            "element_ids": record.get("element_ids", []),
            "content_types": record.get("content_types", []),
            "figure_ids": record.get("figure_ids", []),
            "table_ids": record.get("table_ids", []),
            "formula_ids": record.get("formula_ids", []),
            "image_paths": record.get("image_paths", []),
            "evidence": record.get("evidence", []),
            "text": record.get("retrieval_text") or record.get("text") or "",
        })

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    for rank, result in enumerate(reranked[:top_k], start=1):
        result["rerank_rank"] = rank

    return reranked[:top_k]


In [9]:
# 09 — Run complete retrieval + reranking pipeline

query = "How does portfolio optimization balance expected return and risk?"

dense_results = dense_retrieve(query, DENSE_TOP_K)
bm25_results = bm25_retrieve(query, BM25_TOP_K)

hybrid_candidates = reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    top_k=HYBRID_TOP_K,
    rrf_k=RRF_K,
)

reranked_results = rerank_candidates(
    query,
    hybrid_candidates,
    top_k=FINAL_TOP_K,
)

print("Query:", query)
print("Dense candidates:", len(dense_results))
print("BM25 candidates:", len(bm25_results))
print("Hybrid candidates:", len(hybrid_candidates))
print("Final reranked results:", len(reranked_results))


Query: How does portfolio optimization balance expected return and risk?
Dense candidates: 20
BM25 candidates: 20
Hybrid candidates: 10
Final reranked results: 5


In [10]:
# 10 — Inspect final reranked results

for result in reranked_results:
    print("=" * 100)
    print("Rerank:", result["rerank_rank"])
    print("Chunk:", result["chunk_id"])
    print("Cross-Encoder score:", round(result["rerank_score"], 4))
    print("RRF score:", round(result["rrf_score"], 6))
    print("Dense rank:", result["dense_rank"])
    print("BM25 rank:", result["bm25_rank"])
    print("Source:", result["source_file"])
    print("Pages:", result["pages"])
    print("Section:", result["section_path"])
    print("Content types:", result["content_types"])
    print("Text:", result["text"][:700])


Rerank: 1
Chunk: chunk_3242642c4df23aa2
Cross-Encoder score: 0.9978
RRF score: 0.032018
Dense rank: 1
BM25 rank: 4
Source: 2605.28853v1.pdf
Pages: [2]
Section: ['A. Problem Statement and Preliminaries']
Content types: ['text']
Text: Portfolio optimization is a mathematical framework used to determine the optimal allocation of capital across a set of financial assets. Its primary objective is to maximize the expected return for a specific level of risk, or conversely, to minimize risk for a target level of return. Consider a collection of N assets. Let Rt ∈ RN denote the vector of daily returns of all assets at time t. At the rebalancing date, a neural network produces a vector of portfolio allocation weights wt ∈ RN satisfying
Rerank: 2
Chunk: chunk_da8bb856135d3782
Cross-Encoder score: 0.7559
RRF score: 0.030579
Dense rank: 8
BM25 rank: 3
Source: 2605.29413v1.pdf
Pages: [4]
Section: ['1. Introduction']
Content types: ['text']
Text: The problem of portfolio optimization has been a stap

In [11]:
# 11 — Retrieval → reranking QA

assert len(hybrid_candidates) <= HYBRID_TOP_K
assert len(reranked_results) <= FINAL_TOP_K

metadata_ids = set(metadata_by_id)

for result in reranked_results:
    assert result["chunk_id"] in metadata_ids
    assert result["rerank_rank"] >= 1
    assert result["text"].strip()

print("Reranking QA: PASS")
print("Hybrid candidates:", len(hybrid_candidates))
print("Final reranked chunks:", len(reranked_results))


Reranking QA: PASS
Hybrid candidates: 10
Final reranked chunks: 5


In [12]:
# 12 — Save reranking configuration

RERANK_CONFIG_PATH = VECTOR_STORE_DIR / "reranking_config.json"

with open(RERANK_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "embedding_model": EMBEDDING_MODEL_NAME,
            "reranker_model": RERANKER_MODEL_NAME,
            "dense_top_k": DENSE_TOP_K,
            "bm25_top_k": BM25_TOP_K,
            "hybrid_top_k": HYBRID_TOP_K,
            "final_top_k": FINAL_TOP_K,
            "rrf_k": RRF_K,
        },
        f,
        indent=2,
    )

print("Saved:", RERANK_CONFIG_PATH)


Saved: ..\data\vector_store\reranking_config.json


## Notebook 05 → Notebook 06 contract

```text
Dense Retrieval
      +
BM25
      ↓
     RRF
      ↓
Top 10 Hybrid Candidates
      ↓
Cross-Encoder Reranking
      ↓
Top 5 High-Relevance Chunks
      ↓
Notebook 06:
RAG / Context Assembly / Graph-aware retrieval
```

Every final result retains:

- `chunk_id`
- `source_file`
- `section_path`
- `pages`
- `element_ids`
- `figure_ids`
- `table_ids`
- `formula_ids`
- `image_paths`
- `evidence`

So reranking improves relevance **without breaking provenance and traceability**.
